In [73]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd

In [61]:
path_to_apples = '../plantvillage-dataset/leaf_grouping/filtered_leafmaps/Apple___Apple_scab.csv'
path_to_potatoes = '../plantvillage-dataset/leaf_grouping/filtered_leafmaps/Potato___healthy.csv'
path_to_tomatoes = '../plantvillage-dataset/leaf_grouping/filtered_leafmaps/Tomato___Early_blight.csv'

apple_labels = pd.read_csv(path_to_apples)
potatoes_labels = pd.read_csv(path_to_potatoes)
tomatoes_labels = pd.read_csv(path_to_tomatoes)

In [62]:
path_to_apple_photos = Path('../plantvillage-dataset/raw/color/Apple___Apple_scab')
path_to_potatoes_photos = Path('../plantvillage-dataset/raw/color/Potato___healthy')
path_to_tomatoes_photos = Path('../plantvillage-dataset/raw/color/Tomato___Early_blight')

apple_count = sum(1 for item in path_to_apple_photos.iterdir() if item.is_file())
potato_count = sum(1 for item in path_to_potatoes_photos.iterdir() if item.is_file())
tomato_count = sum(1 for item in path_to_tomatoes_photos.iterdir() if item.is_file())

In [63]:
paths_to_parse = [
    path_to_apple_photos,
    path_to_potatoes_photos,
    path_to_tomatoes_photos
]

labels = [
    apple_labels,
    potatoes_labels,
    tomatoes_labels
]

class_names = ["Apple", "Potatoes", "Tomatoes"]

records = []

for path, current_labels, class_name in zip(
    paths_to_parse, labels, class_names
):
    print(f"Working on {class_name}")

    for file_path in path.iterdir():
        if not file_path.is_file():
            continue

        f = file_path.name

        # Split only once, in case "___" occurs elsewhere
        parts = f.split("___", 1)

        if len(parts) != 2:
            print(f"Skipping unexpected filename: {f}")
            continue

        f_trunc = parts[1]

        match = current_labels.loc[
            current_labels["File Name"] == f_trunc,
            "Leaf #"
        ]

        if len(match) == 0:
            print(f"No label found for: {f_trunc}")
            continue

        if len(match) > 1:
            print(f"Multiple labels found for: {f_trunc}")
            continue

        leaf_id = match.iloc[0]

        records.append({
            "plant name": class_name,
            "file path": f,
            "leaf id": leaf_id
        })

df = pd.DataFrame(records)

Working on Apple
Working on Potatoes
Working on Tomatoes


In [164]:
apples = df[df["plant name"] == "Apple"]

apples_train, apple_test = train_test_split(apples, test_size=0.1, random_state=42)
apples_train, apple_validate = train_test_split(apples_train, test_size=0.111112, random_state=42)

apples_train['split'] = 'training'
apple_validate['split'] = 'validation'
apple_test['split'] = 'test'

training_id = apples_train['leaf id']
C_val_apple = apple_validate["leaf id"].isin(training_id).sum()/len(apple_validate) * 100
C_test_apple = apple_test["leaf id"].isin(training_id).sum()/len(apple_test) * 100

apples = pd.concat([apples_train, apple_validate, apple_test])
split_counts = apples.groupby('leaf id')['split'].nunique()
print(split_counts.value_counts().sort_index())
nonsingle_splits_apple = 0

for val in split_counts:
    if val > 1:
        nonsingle_splits_apple += 1

unique_apple_id = apples['leaf id'].nunique()
G_apple = nonsingle_splits_apple/unique_apple_id * 100

split
1    28
2    62
3    18
Name: count, dtype: int64


In [165]:
potato = df[df["plant name"] == "Potatoes"]

potato_train, potato_test = train_test_split(potato, test_size=0.1, random_state=42)
potato_train, potato_validate = train_test_split(potato_train, test_size=0.111112, random_state=42)

potato_train['split'] = 'training'
potato_validate['split'] = 'validation'
potato_test['split'] = 'test'

training_id = potato_train['leaf id']
C_val_potato = potato_validate["leaf id"].isin(training_id).sum()/len(potato_validate) * 100
C_test_potato = potato_test["leaf id"].isin(training_id).sum()/len(potato_test) * 100

potato = pd.concat([potato_train, potato_validate, potato_test])
split_counts = potato.groupby('leaf id')['split'].nunique()
print(split_counts.value_counts().sort_index())
nonsingle_splits_potato = 0

for val in split_counts:
    if val > 1:
        nonsingle_splits_potato += 1

unique_potato_id = potato['leaf id'].nunique()
G_potato = nonsingle_splits_potato/unique_potato_id * 100

split
1    13
2    23
3     2
Name: count, dtype: int64


In [166]:
tomato = df[df["plant name"] == "Tomatoes"]

tomato_train, tomato_test = train_test_split(tomato, test_size=0.1, random_state=42)
tomato_train, tomato_validate = train_test_split(tomato_train, test_size=0.111112, random_state=42)

tomato_train['split'] = 'training'
tomato_validate['split'] = 'validation'
tomato_test['split'] = 'test'

training_id = tomato_train['leaf id']
C_val_tomato = tomato_validate["leaf id"].isin(training_id).sum()/len(tomato_validate) * 100
C_test_tomato = tomato_test["leaf id"].isin(training_id).sum()/len(tomato_test) * 100

tomato = pd.concat([tomato_train, tomato_validate, tomato_test])
split_counts = tomato.groupby('leaf id')['split'].nunique()
print(split_counts.value_counts().sort_index())
nonsingle_splits_tomato = 0

for val in split_counts:
    if val > 1:
        nonsingle_splits_tomato += 1

unique_tomato_id = tomato['leaf id'].nunique()
G_tomato = nonsingle_splits_tomato/unique_tomato_id * 100

split
1     93
2    137
3     20
Name: count, dtype: int64


In [162]:
results = pd.DataFrame(data={'Class': ['Apple', 'Potatoes', 'Tomatoes'], 'C validate': [C_val_apple, C_val_potato, C_val_tomato], 'C test': [C_test_apple, C_test_potato, C_test_tomato], "G Val": [G_apple, G_potato, G_tomato]})

In [163]:
results

,Class,C validate,C test,G Val
0,Apple,100.0,100.0,74.074074
1,Potatoes,100.0,100.0,65.789474
2,Tomatoes,100.0,100.0,62.800000


In [158]:
apples['leaf id'].nunique()

108